In [1]:
import sys
import pandas as pd
import numpy as np

sys.path.append('.')

from AI_algorithms.utils.data import parse_table
from AI_algorithms.ml.supervised.tree import information_gain
from AI_algorithms.ml.supervised.bayesian import GaussianNB
from AI_algorithms.ml.unsupervised.kmeans import KMeans

Conjunto de datos

In [2]:
data = '''
id x0 x1 x2 C
0 0 0 1 1
1 1 0 1 1
2 0 0 0 1
3 1 0 0 0
4 0 0 0 0
5 1 1 0 0
6 0 1 0 0
7 0 0 1 0
'''

df = parse_table(data, index=True, target=True, columns=True,  target_name='C').astype(int)
df.head()

,x0,x1,x2,C
id,,,,
0,0,0,1,1
1,1,0,1,1
2,0,0,0,1
3,1,0,0,0
4,0,0,0,0


Calcule la ganancia de información de $x_0$

In [3]:
X = df.drop(columns=['C'])
y = df['C'].values

gain = information_gain(X.values,y)
for i, col in enumerate(X.columns):
    print(f'Ganancia de información de {col}: {gain[i]:.4F}')

Ganancia de información de x0: 0.0032
Ganancia de información de x1: 0.2044
Ganancia de información de x2: 0.1589


Entrene un clasificador Bayesiano ingenuo.

Dado un ejemplo x = (1, 1, 0)

Calcule la probabilidad P(C=0|x):

In [4]:
X = df.drop(columns=['C'])
y = df['C'].values

gnb = GaussianNB()
gnb.fit(X, y, discrete=['x0', 'x1', 'x2'])

sample = np.array([[1, 1, 0]])
probs = gnb.predict_proba(sample)
print(f'P(C=0 | x0=1, x1=1, x2=0) = {probs[0][0]:.4F}')
print(f'P(C=1 | x0=1, x1=1, x2=0) = {probs[0][1]:.4F}')

P(C=0 | x0=1, x1=1, x2=0) = 1.0000
P(C=1 | x0=1, x1=1, x2=0) = 0.0000


Considere el conjunto de datos de la primera pregunta y asuma que hay un noveno dato desconocido (id=8, este dato tiene valores continuos entre 0 y 1). Si se hace una iteración de k-means con centroides iniciales correspondientes a los 3 primeros datos (ids 0, 1 y 2), los siguientes son los centroides resultantes:

C0 = (0,0, 0,0, 1,0, 0,5)

C1 = (0,885, 0,33, 0,4425, 0,3375)

C2 = (0,0, 0,3333333333333333, 0,0, 0,3333333333333333)

El ejemplo desconocido es asignado al cluster 1.
En una iteración del algoritmo k-means se asignan los ejemplos y se recalculan los centroides. Si un cluster queda vacio (ningún ejemplo es asignado al centroide correspondiente) asuma que el centroide no cambia.

¿Cuál es el valor de la coordenada 0 del ejemplo desconocido? (de su respuesta con 3 valores decimales)

In [5]:
X = df.drop(columns=['C']).values
kmeans = KMeans(n_clusters=3, max_iter=1, tol=1e-4, )
kmeans.fit(X, centroids=X[:3], verbose=False)
clusters = kmeans.predict(X)
print(f'clusters: {clusters}')

clusters: [0 1 2 1 2 1 2 0]


In [6]:
def update_centroids(data, assigned):
    centroids = []
    for i in range(data.shape[1]):
        cluster_data = data[assigned == i]
        if len(cluster_data) > 0:
            centroids.append(cluster_data.mean(axis=0))
        else:
            centroids.append(np.zeros(data.shape[1]))
    return np.array(centroids)

new_sample = np.zeros((1,3))
new_assigned = np.append(clusters, 1)
new_X = np.vstack((X, new_sample))

def funOptimize(X, assigned, index=(0,0), value=0, value_index=(0,0), target_value=0):
    X[value_index[0], value_index[1]] = value
    new_centroids = update_centroids(X, assigned)
    return new_centroids[index[0], index[1]] - target_value

a,b = -1, 1
t = (1,0)                  #Asignado al cluster 1 coordenada 0
value_index = (len(X), 0)  #Nueva muestra en la última fila, coordenada 0
target_value = 0.85       # Valor objetivo para la coordenada 0 del cluster 1
while funOptimize(new_X, new_assigned, index=t,value_index=value_index,value=a, target_value=target_value) * \
      funOptimize(new_X, new_assigned, value_index=value_index, index=t, value=b, target_value=target_value) >= 0:
    a*=2
    b*=2
print(f'Valor mínimo: {a}, Valor máximo: {b}')

while abs(funOptimize(new_X, new_assigned, index=t, value_index=value_index, value=((a+b)/2), target_value=target_value)) > 1e-5:
    if funOptimize(new_X, new_assigned, index=t, value_index=value_index, value=((a+b)/2), target_value=target_value) > 0:
        b = (a+b)/2
    else:
        a = (a+b)/2
        
val = funOptimize(new_X, new_assigned, index=t, value_index=value_index, value=((a+b)/2), target_value=target_value) + target_value
print(f'Resultado de la bisección: {((a+b)/2):.4F}\n{val}')

Valor mínimo: -1, Valor máximo: 1
Resultado de la bisección: 0.4000
0.850006103515625
